# Cleaning Decisions Explained: How to Understand and Defend Each One

`03_data_cleaning_review.ipynb` found the data problems and proposed actions C1–C6. This notebook takes those decisions **one at a time, in order**, so you can explain each in your own words and answer a challenge to it.

## Natural-language request

> I need to understand and defend every decision I take with respect to data cleaning. Explain every prospective decision sequentially, such that I can explain, understand and defend any decision I have taken. Create another notebook which does that.

## How each decision is laid out

Every decision uses the same eight parts, so you always know where to look:

1. **The question**: what is being decided, in one line.
2. **Evidence**: a code cell that re-computes the facts, so you can say "I checked".
3. **Options considered**: every reasonable choice, including the ones rejected and why.
4. **Recommendation**, and which notebook 03 proposal (C1–C6) and notebook 02 gate (D1–D4) it maps to.
5. **Reasoning chain**: numbered steps from facts to conclusion. If someone attacks the conclusion, find the step they disagree with.
6. **Challenges and answers**: the questions a reviewer is most likely to ask.
7. **What would change this decision**: the evidence that would make you switch. Naming it shows the decision is reasoned, not stubborn.
8. **Say it in one sentence**: a short answer you can give aloud.

The recommendations are proposals. **Your decisions are still recorded in section 10 of notebook 03**; this notebook changes no data and records no choice.

## Ground rules (same as notebook 03)

- The raw file is never changed. This notebook builds no cleaned table.
- Any cell that reads `is_canceled`, `reservation_status` or `reservation_status_date` uses **train only**. Choose and proof are used only for structural counts (no outcome column).
- `src/contract.py` is used unchanged.

## Order of the decisions

The order goes from the decisions with the most consequence to the least, then ends with the one feature decision.

| Step | Decision | Notebook 03 | Does it touch the MVP? |
|---|---|---|---|
| 0 | Principles every decision rests on | — | Yes: they set the bar for all others |
| 1 | Exact repeated rows: keep or remove? | C5 | **Yes**: changes the training base rate and the scored book |
| 2 | Zero-night (same-day) bookings | C5 | **Yes**: changes the scored book and stay value V |
| 3 | Zero-guest rows, room type `P`, children without adults | C5 | Slightly: a few rows in the scored book |
| 4 | Bookings with 10 or more adults | C5 | Barely: 13 train rows |
| 5 | ADR outliers (5,400 and -6.38) | C5 | Barely: moves V on two nights |
| 6 | `meal = Undefined` | C1 | No: `meal` is not a candidate input |
| 7 | NULL `agent` / `company` and decimal IDs | C2 | No |
| 8 | Missing `country`; `CN` vs `CHN` | C3 | No: `country` is Bin 3 |
| 9 | 4 missing `children` values | C4 | No |
| 10 | Things checked that need no action | — | No |
| 11 | `previous_cancellations` drift | C6 | **Yes**: one of the four candidate inputs |
| 12 | How the decisions fit together, and what to record | — | — |

**Tip for a defence:** spend your time on steps 1, 2 and 11. Steps 6–9 are tidy-ups. If a reviewer pushes hard on them, the honest answer is "these do not affect the current model; I chose the option the source supports, and the other option would also have been acceptable."

## Setup

Same loading code as notebook 03. `nights` is the contract's definition; `guests` is used only for validity checks.

In [1]:
from pathlib import Path
import sys

# Find the repository so this works from the repo folder or notebooks/.
repo_root = None
for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "src" / "data.py").exists():
        repo_root = folder
        break

if repo_root is None:
    raise FileNotFoundError("Could not find the project src/ folder.")

source_folder = str(repo_root / "src")
if source_folder not in sys.path:
    sys.path.insert(0, source_folder)

import contract
import data
import pandas as pd

raw_bookings = data.read_raw()
source_columns = list(raw_bookings.columns)
outcome_columns = ["is_canceled", "reservation_status", "reservation_status_date"]
structural_columns = [column for column in source_columns if column not in outcome_columns]

bookings = raw_bookings.copy()
bookings["arrival_date"] = data.arrival_date(bookings)
bookings["nights"] = bookings["stays_in_weekend_nights"] + bookings["stays_in_week_nights"]
bookings["guests"] = bookings["adults"] + bookings["children"].fillna(0) + bookings["babies"]

train, choose, proof = contract.split_periods(bookings)
splits = {"train": train, "choose": choose, "proof": proof}


def count_by_split(checks):
    """Count structural checks in each period without reading outcome columns."""
    rows = []
    for check_name, check in checks.items():
        row = {}
        for period_name, period in splits.items():
            row[period_name] = int(check(period).sum())
        rows.append(pd.Series(row, name=check_name))
    return pd.DataFrame(rows)


print(f"Rows: {len(raw_bookings):,}; source columns: {len(source_columns)}")
print("Rows per period:", {name: len(period) for name, period in splits.items()})

Rows: 119,390; source columns: 32
Rows per period: {'train': 49157, 'choose': 29546, 'proof': 40687}


## Step 0. The principles every decision rests on

Most challenges to a single decision are really challenges to one of these principles. If you can defend these five, the individual decisions follow.

**P1. The raw data is evidence, so it stays unchanged.**
`src/data.py` hands over the file "exactly as exported" and asks that every cleaning choice be logged with its reason and row count. Any change is made on a copy and is reversible.

**P2. The bar for changing data.**
- *Change a value* only when a source defines what the value should be (for example, the dataset article says `Undefined` and `SC` both mean "no meal package").
- *Remove a row* only when a source rule proves the row is an error.
- "It looks odd" is a reason to **investigate and flag**, not to delete. Without a booking ID, most odd rows cannot be proved wrong.

**P3. The scored book is fixed by the contract.**
The contract scores one hotel on one arrival night, and "a night's book is every booking due to arrive that night". If we delete a row, we change the book the client's rule is scored on. That would quietly change the contract, which rule 1 forbids. Rule 5 also requires every candidate and the flat-rule baseline to be scored on the *same* hotel-nights.

**P4. The training data must look like the data it will be scored on.**
A model learns base rates from train. If cleaning removes rows from train but those rows still appear in the scored book, the model is trained on one population and judged on another.

**P5. Outcomes never drive cleaning.**
A row is not removed *because* it cancelled. Removing rows according to their outcome would bias the target the model learns. Outcome columns are read only on train, and only to understand a group, never to choose which rows to keep (rules 2 and 3).

**Why add `flag_` columns at all?** A flag records an observation without changing anything. It costs nothing, keeps every odd row traceable, and lets you run a sensitivity check later on choose (for example, "does the result change if flagged nights are reported separately?"). That is better than deleting the rows, which would be irreversible, and better than ignoring them, which would leave no record.

## Step 1. Exact repeated rows: keep or remove?

**The question.** 16,291 train rows exactly repeat an earlier row in all 32 columns. Are they export errors to delete, or real separate bookings to keep?

In [2]:
# Train only: this cell reads is_canceled.
full_repeat = train[source_columns].duplicated()               # later copies, all 32 columns
full_group = train[source_columns].duplicated(keep=False)      # every row in a 32-column identical group
look_alike_group = train[structural_columns].duplicated(keep=False)  # identical except outcome columns

print(f"Later copies (all 32 columns): {full_repeat.sum():,}")
print(f"Rows in an identical group (all 32 columns): {full_group.sum():,}")
print(f"Rows in an identical group (non-outcome columns only): {look_alike_group.sum():,}")

# Key test: do bookings that look identical before the outcome always end the same way?
groups = (
    train[look_alike_group]
    .groupby(structural_columns, dropna=False)["is_canceled"]
    .agg(["nunique", "size"])
)
mixed = groups[groups["nunique"] > 1]
print(f"\nLook-alike groups: {len(groups):,}")
print(f"  groups whose bookings had DIFFERENT outcomes: {len(mixed):,} ({mixed['size'].sum():,} rows)")

print("\nLater copies by market segment (top 3):")
print(train.loc[full_repeat, "market_segment"].value_counts().head(3).to_string())

rate_all = train["is_canceled"].mean()
rate_dedup = train.loc[~full_repeat, "is_canceled"].mean()
print(f"\nTrain cancellation rate: {rate_all:.1%} kept; {rate_dedup:.1%} if later copies were dropped")

print("\nShare of each period's book that repeats an earlier row (non-outcome columns):")
for name, period in splits.items():
    print(f"  {name}: {period[structural_columns].duplicated().mean():.1%}")

Later copies (all 32 columns): 16,291
Rows in an identical group (all 32 columns): 19,763
Rows in an identical group (non-outcome columns only): 19,982

Look-alike groups: 3,459
  groups whose bookings had DIFFERENT outcomes: 125 (1,128 rows)

Later copies by market segment (top 3):
market_segment
Groups           8047
Offline TA/TO    5964
Online TA        1273

Train cancellation rate: 35.8% kept; 22.2% if later copies were dropped

Share of each period's book that repeats an earlier row (non-outcome columns):
  train: 33.6%


  choose: 23.1%
  proof: 22.5%


**Options considered**

| Option | What it does | Verdict |
|---|---|---|
| A. Drop every later copy | Keeps one row per identical pattern | **Rejected.** No booking ID proves the copies are errors (P2). Drops the train rate from 35.8% to 22.2% while about 23% of the choose and proof book are still repeats, so the model would under-predict cancellations on the book it is scored on (P4). Changes the scored book (P3). |
| B. Drop copies in train only, keep them in the scored book | Cleans the training data, leaves the contract alone | **Rejected.** Avoids changing the book, but creates exactly the train/score mismatch in P4. |
| C. Down-weight copies when fitting | Each group counts less | **Not a cleaning step.** This is a modelling option; it could be tested on choose in Step 5 if wanted. |
| D. **Keep every row and flag identical groups** | No row removed; `flag_identical_group` marks them | **Recommended.** |

**Recommendation:** Option D. Notebook 03 proposal **C5**; answers notebook 02 gate **D1**.

**Reasoning chain**

1. The file has no booking or reservation ID, so "identical row" does not mean "same booking".
2. The repeats cluster in `Groups` and `Offline TA/TO`, in identical groups of up to 150 rows. That fits a multi-room group booking exported one row per room.
3. Some groups that look identical before the outcome **ended differently** (see the "DIFFERENT outcomes" line above). A copy of one booking cannot both cancel and check out, so at least those groups are separate bookings.
4. The repeats are strongly linked to the target: dropping them moves the train rate from 35.8% to 22.2%. That is not a small tidy-up; it changes what the model learns.
5. The choose and proof books still contain repeats (about 23%). The contract scores those books as they are.
6. Therefore removing repeats would train the model on a different population from the one it is scored on and would change the scored book. Keep them and flag them.

**Challenges and answers**

- *"Duplicates are textbook dirty data. Why keep them?"* Textbook removal assumes a key that proves two rows are the same record. We have no such key, and some look-alike groups have different outcomes, which a copied record cannot have.
- *"Keeping them inflates the cancellation rate."* The rate with them is the rate of the book being scored. Removing them would deflate it relative to the scored book.
- *"Train has more repeats (33.6%) than choose and proof (about 23%). Isn't that a problem?"* Yes: that is drift in the mix of bookings, and it is real. It is handled by choosing on choose, and the contract allows refitting on train + choose before the single proof run. Deleting rows would not fix drift; it would just remove a different amount.
- *"Could copies leak between training and evaluation?"* No. Identical rows share their arrival date, so every copy falls in the same contract period. The time split keeps them together.
- *"Is anything still wrong after keeping them?"* Yes, and it should be said: rows in a group tend to cancel together. `contract.nightly_table`'s `spread` assumes bookings cancel independently, so it will understate night-to-night uncertainty. That matters for choosing the buffer on choose, not for cleaning.
- *"Why do the 32-column count (19,763) and the flag count (19,982) differ?"* The flag compares only non-outcome columns, so it never reads an outcome. The extra 219 train rows look identical before the outcome but differ in outcome or status date; they are part of the evidence in step 3 of the reasoning chain.

**What would change this decision:** a booking or reservation ID, or a client statement that the export duplicated rows, showing which rows are true copies. Then you would remove the proved copies, in **both** train and the scored book, and log the count.

**Say it in one sentence:** "Without a booking ID I can't prove the repeats are errors, some look-alike rows had different outcomes, and removing them would cut the training cancellation rate from 36% to 22% while they're still about a quarter of the book we're scored on, so I kept and flagged them."

## Step 2. Zero-night (same-day) bookings

**The question.** 715 bookings have `nights = 0`. They never occupy a room overnight. Should they leave the training table or the scored book?

In [3]:
zero_night_checks = {"zero nights": lambda d: d["nights"].eq(0)}
print("Zero-night bookings by period (no outcome read):")
print(count_by_split(zero_night_checks).to_string())

# Train only: what happened to these bookings?
print("\nTrain zero-night bookings by final status:")
print(train.loc[train["nights"].eq(0), "reservation_status"].value_counts().to_string())

# Structural only: how much would removing them move the stay value V on the nights they sit on?
print("\nEffect of removing them on V = mean(adr x nights), per hotel-night:")
for name, period in splits.items():
    stay = period["adr"] * period["nights"]
    keys = [period["hotel"], period["arrival_date"]]
    v_kept = stay.groupby(keys).mean()
    has_stay = period["nights"].gt(0)
    v_removed = stay[has_stay].groupby([period["hotel"][has_stay], period["arrival_date"][has_stay]]).mean()
    change = (v_removed - v_kept.reindex(v_removed.index))
    changed = change[change.abs() > 1e-9]
    print(f"  {name}: {len(changed)} of {len(v_kept)} nights change; "
          f"average rise EUR {changed.mean():.2f}, largest EUR {changed.max():.2f}")

Zero-night bookings by period (no outcome read):
             train  choose  proof
zero nights    371     200    144

Train zero-night bookings by final status:
reservation_status
Check-Out    354
No-Show       10
Canceled       7

Effect of removing them on V = mean(adr x nights), per hotel-night:
  train: 231 of 732 nights change; average rise EUR 9.85, largest EUR 71.58
  choose: 140 of 368 nights change; average rise EUR 8.34, largest EUR 62.61
  proof: 93 of 486 nights change; average rise EUR 8.34, largest EUR 38.18


**Options considered**

| Option | Verdict |
|---|---|
| A. Remove from training and the scored book | **Rejected.** The contract defines the book as every booking due to arrive; a same-day stay is due to arrive (P3, rule 1). It would also raise V on about a third of choose nights, changing the cost of every empty room on those nights. |
| B. Remove from training only | **Rejected.** The model would never learn that same-day bookings rarely cancel, yet they still sit in the scored book (P4). |
| C. **Keep and flag (`flag_zero_nights`)** | **Recommended.** Notebook 03 **C5**; notebook 02 gate **D3**. |

**Reasoning chain**

1. In train, 354 of 371 zero-night bookings checked out. They are real, completed stays, not broken records.
2. The contract's book is "every booking due to arrive that night"; it does not exclude same-day stays.
3. Their stay value is 0, which pulls a night's V down. Removing them would raise V on the affected nights by about EUR 8–10 on average (up to about EUR 72). That is a change to the scoring, not cleaning.
4. Whether a same-day stay *should* count toward a night's oversell is a business-definition question for the client, not something we can settle from the data.

**Challenges and answers**

- *"A zero-night stay can't use a room overnight, so it's an error."* It is unusual, not an error: the hotel recorded check-outs. Day-use rooms exist. No source says these are invalid.
- *"They distort V."* They do, slightly, and V is defined by the contract. Changing V would mean changing the client's scoring rule, which we are not allowed to do. We raise it as a question for the client instead.
- *"They rarely cancel (4.6% in train). Isn't keeping them helping the model?"* We keep them because the source and the contract say they belong, not because of their outcome (P5). The low rate is simply true of these bookings, and the scored book contains them.

**What would change this decision:** the client saying same-day stays are not part of overnight inventory. Then the contract's book definition itself would need the client's sign-off to change, and it should be applied to every candidate and to the baseline alike.

**Say it in one sentence:** "Same-day bookings are real stays (354 of 371 in train checked out) and the contract counts every booking due to arrive, so removing them would change the client's scoring; I flagged them and logged the definition as a client question."

## Step 3. Zero-guest rows, room type `P`, and children without adults

**The question.** Some rows record no guests, a placeholder room type `P`, or children with no adult. Are they errors to remove?

In [4]:
odd_checks = {
    "zero guests": lambda d: d["guests"].eq(0),
    "reserved room type P": lambda d: d["reserved_room_type"].eq("P"),
    "room type P with zero guests": lambda d: d["reserved_room_type"].eq("P") & d["guests"].eq(0),
    "children or babies but 0 adults": lambda d: d["adults"].eq(0) & d["guests"].gt(0),
}
print("Counts by period (no outcome read):")
print(count_by_split(odd_checks).to_string())

# Train only: are these real stays?
print("\nTrain zero-guest bookings by final status:")
print(train.loc[train["guests"].eq(0), "reservation_status"].value_counts().to_string())
print("\nTrain children-without-adults bookings by final status:")
no_adult = train["adults"].eq(0) & train["guests"].gt(0)
print(train.loc[no_adult, "reservation_status"].value_counts().to_string())

Counts by period (no outcome read):
                                 train  choose  proof
zero guests                         49      64     67
reserved room type P                 1       5      6
room type P with zero guests         1       5      6
children or babies but 0 adults     72      88     63

Train zero-guest bookings by final status:
reservation_status
Check-Out    45
Canceled      3
No-Show       1

Train children-without-adults bookings by final status:
reservation_status
Check-Out    51
Canceled     20
No-Show       1


**Options considered**

| Option | Verdict |
|---|---|
| A. Remove all of them | **Rejected.** Most zero-guest rows in train checked out (45 of 49), so a guest count was missed, not the booking. Removing them removes real arrivals from the book (P3). |
| B. Remove only the 12 room-type `P` rows | **Defensible but rejected.** They look like placeholders (0 guests, ADR 0, mostly 0 nights). But no source rule defines `P` as invalid, and with 12 rows across 3 periods the effect on any night's book is one or two bookings. |
| C. Impute adult counts | **Rejected.** No source tells us the true number, and guest counts are not candidate inputs. |
| D. **Keep and flag** (`flag_zero_guests`, `flag_room_type_p`, `flag_children_without_adults`) | **Recommended.** Notebook 03 **C5**; notebook 02 gate **D3**. |

**Reasoning chain**

1. The odd field is the guest count, not the booking's existence: most of these rows checked out.
2. Guest counts are Bin 2 and not candidate inputs, so a wrong count does not reach the model.
3. Each row is still a booking due to arrive, so it belongs in the scored book.
4. The counts are small, so keeping them cannot distort the training base rate much.

**Challenges and answers**

- *"A booking with zero guests is impossible."* The record of guests is incomplete; the stay happened. We do not delete a real stay because one field is wrong.
- *"Room type `P` is clearly a placeholder. Why keep it?"* I agree it probably is, and it is flagged. Twelve rows can't move results, and removing them would need a rule we don't have. If the client confirms `P` is a system placeholder, I would remove those 12 from both training and the book and log it.

**What would change this decision:** a client or system document stating that `P`, or zero-guest records, are test or placeholder entries.

**Say it in one sentence:** "Most of these rows are completed stays with a bad guest count, guest counts aren't model inputs, and nothing proves they're invalid bookings, so I flagged rather than removed them."

## Step 4. Bookings with 10 or more adults

**The question.** 13 train bookings have 10–55 adults on one row. Remove them as impossible single-room bookings?

In [5]:
large = train["adults"].ge(10)
print(f"Train rows with 10+ adults: {large.sum()}; in choose: {choose['adults'].ge(10).sum()}; in proof: {proof['adults'].ge(10).sum()}")
print(train.loc[large, ["hotel", "arrival_date", "adults", "nights", "adr", "market_segment", "is_canceled"]]
      .describe(include="all").loc[["count", "unique", "top", "min", "max"]].to_string())

# Train only: how much do 13 rows move the base rate?
print(f"\nTrain cancellation rate: {train['is_canceled'].mean():.2%} with them; "
      f"{train.loc[~large, 'is_canceled'].mean():.2%} without them")

Train rows with 10+ adults: 13; in choose: 0; in proof: 0
               hotel         arrival_date  adults  nights   adr market_segment  is_canceled
count             13                   13    13.0    13.0  13.0             13         13.0
unique             1                  NaN     NaN     NaN   NaN              2          NaN
top     Resort Hotel                  NaN     NaN     NaN   NaN         Direct          NaN
min              NaN  2015-09-03 00:00:00    10.0     2.0   0.0            NaN          1.0
max              NaN  2015-10-12 00:00:00    55.0     7.0   0.0            NaN          1.0

Train cancellation rate: 35.77% with them; 35.76% without them


**Options considered**

| Option | Verdict |
|---|---|
| A. Remove the 13 rows | **Rejected.** They change the train rate by about 0.02 percentage points, and the main thing that singles them out is that they all cancelled. Removing rows for that reason is outcome-driven cleaning (P5). |
| B. Cap adults at some value | **Rejected.** Any cap would be arbitrary, and `adults` is not a candidate input. |
| C. **Keep and flag (`flag_ten_plus_adults`)** | **Recommended.** Notebook 03 **C5**; notebook 02 gate **D3**. |

**Reasoning chain**

1. All 13 are Resort Hotel, September–October 2015, ADR 0, and cancelled. They look like group-block placeholders, but no source says so.
2. They exist only in train, so they are never in a scored choose or proof book.
3. Their effect on the base rate is negligible (shown above).
4. Removing them would be justified mainly by their outcome, which P5 forbids.

**Challenges and answers**

- *"55 adults in one room is impossible."* In one room, yes. A row with 55 adults is more likely a block reservation recorded on one line. It is flagged so it can be explained, and it is too small to matter.
- *"They all cancelled, so they'll bias the model."* Thirteen rows out of 49,157 move the rate by about 0.02 points. And deleting rows *because* they cancelled is the bias to avoid.

**What would change this decision:** evidence that block placeholders were exported as bookings. Then remove all rows of that kind by a stated rule, not just these 13.

**Say it in one sentence:** "Thirteen train-only rows that move the base rate by 0.02 points, singled out mainly by their outcome: I flagged them, because removing them for cancelling would be outcome-driven cleaning."

## Step 5. ADR outliers: 5,400 and -6.38

**The question.** One train row has ADR 5,400 (next highest 508). One proof row has ADR -6.38. Correct, cap, or keep?

In [6]:
print("Three largest ADR values in train:")
print(train.nlargest(3, "adr")[["hotel", "arrival_date", "adults", "nights", "adr", "deposit_type"]].to_string())


def same_night(period, row_index):
    """Return every booking on the same hotel and arrival night as the given row."""
    row = period.loc[row_index]
    return period[(period["hotel"] == row["hotel"]) & (period["arrival_date"] == row["arrival_date"])]


top_adr = train["adr"].idxmax()
night = same_night(train, top_adr)
stay = night["adr"] * night["nights"]
print(f"\nADR 5,400 night (train): {len(night)} bookings; V = {stay.mean():.2f} with the row, "
      f"{stay.drop(top_adr).mean():.2f} without it")

negative_adr = proof.index[proof["adr"].lt(0)][0]  # structural columns only; no outcome read
night = same_night(proof, negative_adr)
stay = night["adr"] * night["nights"]
print(f"Negative-ADR night (proof): {len(night)} bookings; V = {stay.mean():.2f} as recorded, "
      f"{stay.where(night['adr'].ge(0), 0).mean():.2f} with ADR set to 0")

Three largest ADR values in train:
              hotel arrival_date  adults  nights      adr deposit_type
48515    City Hotel   2016-03-25       2       1  5400.00   Non Refund
15083  Resort Hotel   2015-07-15       2       1   508.00   No Deposit
90280    City Hotel   2016-06-02       2       3   332.57   No Deposit

ADR 5,400 night (train): 182 bookings; V = 377.73 with the row, 349.99 without it
Negative-ADR night (proof): 83 bookings; V = 336.69 as recorded, 337.46 with ADR set to 0


**Options considered**

| Option | Verdict |
|---|---|
| A. Delete the rows | **Rejected.** Removes a booking from the book to fix one field (P3). |
| B. Set the 5,400 row to a "plausible" value, or cap ADR at a percentile | **Rejected.** No source gives the correct value or a valid maximum; any cap is our invention. |
| C. Set the negative ADR to 0 | **Defensible but rejected.** It changes a proof scoring input to fix a difference of under EUR 1 on one night. |
| D. **Keep the values; flag them** (`flag_adr_above_1000`, `flag_adr_negative`) | **Recommended.** Notebook 03 **C5**; notebook 02 gate **D3**. |

**Reasoning chain**

1. ADR is not a candidate input. It matters only through V in the contract's empty-room cost.
2. The 5,400 row is in train, and train nights are not used to choose methods or buffers, so it changes nothing that is decided or scored.
3. The negative ADR moves its proof night's V by under EUR 1.
4. Correcting either value would mean inventing a number with no source to support it. When the impact is this small, the safe default is to leave the evidence alone and flag it.

**Challenges and answers**

- *"5,400 is obviously a typo."* Probably, and it is flagged. But I can't prove the right value (540? 54?), and it changes nothing scored. I would rather leave a flagged true record than invent a number.
- *"A negative price is impossible."* The article computes ADR from lodging transactions, which can include refunds or adjustments, so a small negative is possible in accounting terms. Either way, its effect is under EUR 1.
- *"Did you look at the proof row's outcome?"* No. Only its hotel, date, guests, nights, ADR, segment, deposit type and room type were shown.

**What would change this decision:** ADR becoming a model input (then extreme values would need a train-fitted treatment), or the hotel confirming the correct values.

**Say it in one sentence:** "ADR isn't a model input, and the two outliers move V by EUR 28 on an unscored train night and under EUR 1 on a proof night, so I flagged them instead of inventing replacements."

## Step 6. `meal = Undefined` (C1)

**The question.** Should `Undefined` be recoded to `SC`?

In [7]:
meal_checks = {
    "meal = Undefined": lambda d: d["meal"].eq("Undefined"),
    "meal = SC": lambda d: d["meal"].eq("SC"),
}
print(count_by_split(meal_checks).to_string())

print("\nMeal codes by hotel (all periods, no outcome read):")
print(pd.crosstab(bookings["meal"], bookings["hotel"]).to_string())

resort = bookings[bookings["hotel"].eq("Resort Hotel")]
print("\nResort Hotel 'no meal' codes by arrival quarter:")
print(pd.crosstab(resort["arrival_date"].dt.to_period("Q"), resort["meal"])[["SC", "Undefined"]].to_string())

                  train  choose  proof
meal = Undefined    548     208    413
meal = SC          2471    3001   5178

Meal codes by hotel (all periods, no outcome read):


hotel      City Hotel  Resort Hotel
meal                               
BB              62305         30005
FB                 44           754
HB               6417          8046
SC              10564            86
Undefined           0          1169

Resort Hotel 'no meal' codes by arrival quarter:
meal          SC  Undefined
arrival_date               
2015Q3         6          0
2015Q4         9        170
2016Q1         7        251
2016Q2        13        127
2016Q3         5         55
2016Q4         7        153
2017Q1        19        283
2017Q2         8         94
2017Q3        12         36


**Options considered**

| Option | Verdict |
|---|---|
| A. Leave `Undefined` as its own category | **Acceptable.** Nothing in the MVP uses `meal`. |
| B. Treat `Undefined` as missing (`NaN`) | **Rejected.** The source says it is not missing: it means "no meal package". |
| C. **Recode `Undefined` to `SC`** | **Recommended.** Notebook 03 **C1**; notebook 02 gate **D2**. |

**Reasoning chain**

1. The dataset article (Antonio et al., 2019) defines the category as "Undefined/SC – no meal package". That is a source definition, which is the bar P2 sets for changing a value.
2. City Hotel records "no meal" as `SC` (10,564 rows) and almost never as `Undefined` (0). Resort uses `Undefined` mostly (1,169) and `SC` rarely (86). The two codes mean the same thing, recorded differently by each hotel's staff.
3. If `meal` is ever used (it is Bin 2), two codes for one meaning would split one group into two, and the split would partly stand for "which hotel", which `hotel` already captures.
4. `SC` is the named code that both hotels use; `Undefined` is ambiguous in plain English. So merge into `SC`.

**Be precise about one detail:** Resort uses *both* codes in most quarters (see the table above). So "each hotel has its own code" is mostly true, not completely. The decision rests on the article's definition, not on that pattern.

**Challenges and answers**

- *"Maybe `Undefined` means 'not recorded'."* The article lists it explicitly as "no meal package", so we follow the source rather than guess.
- *"Did you check cancellation rates before deciding?"* No, deliberately. The recode rests on the source definition; choosing it by outcome would break P5.
- *"Why bother, if `meal` isn't used?"* It is a cheap, source-backed tidy-up that prevents a mistake if Bin 2 fields are used later, for example in the diagnostic leakage run in `docs/COLUMNS.md`. Leaving it would also be acceptable.

**What would change this decision:** a hotel statement that `Undefined` means something other than no meal.

**Say it in one sentence:** "The dataset article defines Undefined and SC as the same 'no meal' category, so I merged them under SC; it doesn't affect the current model."

## Step 7. NULL `agent` / `company`, and IDs stored as decimals (C2)

**The question.** `agent` and `company` are read as decimals (9.0, 240.0) with `NaN` for NULL. Store them as text with NULL as `none`?

In [8]:
id_checks = {
    "agent NULL": lambda d: d["agent"].isna(),
    "company NULL": lambda d: d["company"].isna(),
}
print(count_by_split(id_checks).to_string())

for column in ["agent", "company"]:
    values = bookings[column].dropna()
    print(f"\n{column}: dtype {bookings[column].dtype}; distinct IDs {values.nunique()}; "
          f"smallest {values.min():.0f}; all whole numbers: {bool(values.mod(1).eq(0).all())}")

              train  choose  proof
agent NULL     6960    3716   5664
company NULL  46123   28093  38377

agent: dtype float64; distinct IDs 333; smallest 1; all whole numbers: True

company: dtype float64; distinct IDs 352; smallest 6; all whole numbers: True


**Options considered**

| Option | Verdict |
|---|---|
| A. Leave as decimals with `NaN` | **Acceptable but risky.** A model or encoder could treat ID 240 as "bigger" than ID 9, and many tools drop or impute `NaN`. |
| B. Fill NULL with 0 | **Rejected.** 0 looks like an ID, and the code still treats IDs as numbers. |
| C. Impute NULL with the most common ID | **Rejected.** The source says NULL means "no agent", so imputing would invent an agent. |
| D. **Store as text; NULL becomes `none`** | **Recommended.** Notebook 03 **C2**; notebook 02 gate **D2**. |

**Reasoning chain**

1. The article says NULL in `agent` / `company` is a category meaning "not applicable" (no agent, no company), not a missing value.
2. The IDs are labels. Averages or "greater than" comparisons of IDs are meaningless, so they should be stored as text.
3. Every non-missing ID is a whole number (≥ 1), so converting to text loses nothing, and `none` cannot be confused with a real ID.
4. Neither field is a candidate input, so this only prepares them for later use.

**Challenges and answers**

- *"94% of `company` is NULL. Isn't the column useless?"* Possibly as a predictor, but that is a feature decision. Cleaning only makes the value mean what the source says.
- *"Why not just drop the columns?"* Dropping columns is a feature-selection choice. They are already excluded from the MVP by the column audit.

**What would change this decision:** nothing in the data; only a source saying NULL means "unknown" would make imputation worth considering.

**Say it in one sentence:** "The source says NULL means 'no agent / no company', and IDs are labels, so I stored them as text with an explicit `none`."

## Step 8. Missing `country`, and `CN` vs `CHN` (C3)

**The question.** Label missing countries `unknown`? Merge the 2-letter `CN` into the 3-letter `CHN`?

In [9]:
country_checks = {
    "country missing": lambda d: d["country"].isna(),
    "country = CN": lambda d: d["country"].eq("CN"),
    "country = CHN": lambda d: d["country"].eq("CHN"),
}
print(count_by_split(country_checks).to_string())
print("\nCountry code lengths:", bookings["country"].dropna().str.len().value_counts().sort_index().to_dict())

china_codes = bookings["country"].where(bookings["country"].isin(["CN", "CHN"]))
print("\nCN and CHN by hotel and arrival year (both used side by side):")
print(pd.crosstab([bookings["hotel"], bookings["arrival_date"].dt.year], china_codes).to_string())

                 train  choose  proof
country missing    284     121     83
country = CN       279     362    638
country = CHN      270     295    434

Country code lengths: {2: 1279, 3: 117623}

CN and CHN by hotel and arrival year (both used side by side):


country                    CHN   CN
hotel        arrival_date          
City Hotel   2015           27   51
             2016          467  239
             2017          371  279
Resort Hotel 2015           12   81
             2016           59  270
             2017           63  359

**Options considered**

| Option | Verdict |
|---|---|
| A. Merge `CN` into `CHN` | **Rejected.** Likely right, but unconfirmed. `CN` is the only 2-letter code, and it appears side by side with `CHN` in both hotels and every year, so it is not a simple change of format at some date. Merging would be a guess. |
| B. Leave missing as `NaN` | **Acceptable.** `country` is never a predictor. |
| C. **Label missing `unknown`; keep `CN` and `CHN` separate** | **Recommended.** Notebook 03 **C3**; notebook 02 gate **D2**. |

**Reasoning chain**

1. `country` is Bin 3: the article warns that the hotel may not know a guest's country until check-in. It is never a predictor.
2. An explicit `unknown` label says what the value means and keeps those rows in any count or table.
3. `CN` is probably China's 2-letter ISO code, but nothing in the source confirms it, and both codes are used throughout. P2 sets the bar at a source definition, so do not merge.

**Challenges and answers**

- *"Everyone knows CN is China."* Probably right. But the source does not say so, and merging changes nothing we score. A flagged probability is safer than an unproved merge.
- *"Why clean a column you'll never use?"* It is a minimal label for reporting. Disapproving C3 would also be fine.

**What would change this decision:** the hotel confirming the country-code table.

**Say it in one sentence:** "Country is never a predictor; I labelled missing values `unknown` and didn't merge CN into CHN because no source confirms they are the same."

## Step 9. Four missing `children` values (C4)

**The question.** Fill the 4 missing `children` values, and with what?

In [10]:
missing_children = train["children"].isna()
print(f"Missing children: train {missing_children.sum()}, choose {choose['children'].isna().sum()}, proof {proof['children'].isna().sum()}")
print(f"Share of train rows with 0 children: {train['children'].eq(0).mean():.1%}")

print("\nThe 4 rows (structural columns only):")
print(train.loc[missing_children, ["hotel", "arrival_date", "adults", "babies", "nights", "adr",
                                    "market_segment", "distribution_channel"]].to_string())

undefined_any = train["market_segment"].eq("Undefined") | train["distribution_channel"].eq("Undefined")
print(f"\nTrain rows with an Undefined segment or channel: {undefined_any.sum()}; "
      f"of these, missing children: {(undefined_any & missing_children).sum()}")

Missing children: train 4, choose 0, proof 0
Share of train rows with 0 children: 95.1%

The 4 rows (structural columns only):
            hotel arrival_date  adults  babies  nights   adr market_segment distribution_channel
40600  City Hotel   2015-08-03       2       0       1  12.0      Undefined            Undefined
40667  City Hotel   2015-08-05       2       0       2  12.0         Direct            Undefined
40679  City Hotel   2015-08-05       3       0       2  18.0      Undefined            Undefined
41160  City Hotel   2015-08-13       2       0       7  76.5      Online TA            Undefined



Train rows with an Undefined segment or channel: 5; of these, missing children: 4


**Options considered**

| Option | Verdict |
|---|---|
| A. Remove the 4 rows | **Rejected.** Removes bookings from the book to fix a field that is not a model input (P3). |
| B. Leave as `NaN` | **Acceptable.** The guest count used in the checks already treats it as 0. |
| C. Fill with the mean | **Rejected.** A fractional child count is meaningless. |
| D. **Fill with 0 and keep `flag_children_missing`** | **Recommended.** Notebook 03 **C4**; notebook 02 gate **D2**. |

**Reasoning chain**

1. The 4 rows are City Hotel bookings from early August 2015. All 4 also have `distribution_channel = Undefined`, and 2 have `market_segment = Undefined`: a small cluster of incompletely recorded bookings, not a general problem.
2. 95.1% of train rows have 0 children, so 0 is both the most common value and the median.
3. The flag records that the 0 was filled, so the change is traceable and reversible.
4. `children` is not a candidate input. This keeps the column a whole-number count.

**Challenges and answers**

- *"Filling with 0 is a guess."* It is the most common value and is recorded with a flag. With 4 rows of 49,157, any choice is immaterial, so the decision is about keeping the column clean and the change traceable.

**What would change this decision:** `children` becoming a model input. Then the fill would be fitted on train only, as part of the feature pipeline.

**Say it in one sentence:** "Four train rows from one incompletely recorded cluster, filled with the 95% majority value of 0 and flagged so the fill is traceable."

## Step 10. Things checked that need no action

Being able to say "I checked this and left it alone, because…" is part of a defence. These checks found nothing to clean:

In [11]:
# Train only: target vs status, and status date consistency.
print("Target vs reservation status (train):")
print(pd.crosstab(train["reservation_status"], train["is_canceled"]).to_string())

# Structural, all periods: which week definition does arrival_date_week_number use?
jan1_weekday = pd.to_datetime(bookings["arrival_date_year"].astype(str) + "-01-01").dt.dayofweek
sunday_offset = (jan1_weekday + 1) % 7
sunday_week = (bookings["arrival_date"].dt.dayofyear - 1 + sunday_offset) // 7 + 1
iso_week = bookings["arrival_date"].dt.isocalendar().week.astype(int)
week = bookings["arrival_date_week_number"]
print(f"\nWeek number differs from the ISO week: {week.ne(iso_week).sum():,} rows")
print(f"Week number differs from Sunday-start weeks: {week.ne(sunday_week).sum():,} rows")

print(f"\nmarket_segment = Undefined: {bookings['market_segment'].eq('Undefined').sum()} rows; "
      f"distribution_channel = Undefined: {bookings['distribution_channel'].eq('Undefined').sum()} rows (all train)")

Target vs reservation status (train):


is_canceled             0      1
reservation_status              
Canceled                0  16959
Check-Out           31572      0
No-Show                 0    626



Week number differs from the ISO week: 64,054 rows
Week number differs from Sunday-start weeks: 0 rows



market_segment = Undefined: 2 rows; distribution_channel = Undefined: 5 rows (all train)


| Check | Finding | Why no action |
|---|---|---|
| `is_canceled` vs `reservation_status` | Exact match in train: `Canceled` and `No-Show` are 1; `Check-Out` is 0 | The target means what the contract says. Both remain outcome fields, never predictors (rule 3). |
| Status dates | None before the booking date; no cancellation dated after arrival; 13 early check-outs | Early departure is real behaviour, not an error. |
| `arrival_date_week_number` | Not ISO weeks, but matches Sunday-start weeks exactly | Internally consistent. If a calendar feature is needed, derive it from `arrival_date`. |
| `market_segment` / `distribution_channel` = `Undefined` | 2 and 5 train rows, overlapping the incomplete August 2015 cluster from step 9 | Not candidate inputs, too few to matter, and no source defines what they should be. |

**Say it in one sentence:** "I checked the target against status, the dates, the week numbers and the Undefined segment codes; they are consistent or immaterial, so I changed nothing."

## Step 11. `previous_cancellations`: drift and possible group counting (C6)

**The question.** This is one of the four candidate inputs, and its behaviour changes sharply over time. Change it, drop it, or test it?

This is a **feature** decision, not a value fix, and it is the one most likely to change the model's results.

In [12]:
# Structural, all periods: how common are prior cancellations over time?
had_prior = bookings["previous_cancellations"].gt(0)
print("Share of bookings with prior cancellations, by arrival quarter:")
print(had_prior.groupby(bookings["arrival_date"].dt.to_period("Q")).mean().to_string(float_format="{:.1%}".format))

# Train only: is the link to cancellation stable inside train?
train_prior = train["previous_cancellations"].gt(0)
train_half = pd.Series("2016 H1", index=train.index)
train_half[train["arrival_date"] < "2016-01-01"] = "2015 H2"
print("\nTrain cancellation rate of bookings WITH prior cancellations, by half-year:")
print(train.loc[train_prior].groupby(train_half[train_prior])["is_canceled"].mean().to_string(float_format="{:.1%}".format))

# Train only: where do these rows come from?
in_group = train[structural_columns].duplicated(keep=False)
print(f"\nTrain rows with prior cancellations: {train_prior.sum():,}")
print(f"  in an identical-row group: {in_group[train_prior].mean():.0%}")
print(f"  marked 'not a repeated guest': {train.loc[train_prior, 'is_repeated_guest'].eq(0).mean():.0%}")
print(f"  Non Refund deposit: {train.loc[train_prior, 'deposit_type'].eq('Non Refund').mean():.0%} "
      f"(all train bookings: {train['deposit_type'].eq('Non Refund').mean():.0%})")

Share of bookings with prior cancellations, by arrival quarter:


arrival_date
2015Q3   23.8%
2015Q4   16.0%
2016Q1    8.2%
2016Q2    2.9%
2016Q3    1.3%
2016Q4    1.4%
2017Q1    0.9%
2017Q2    0.6%
2017Q3    0.6%
Freq: Q-DEC

Train cancellation rate of bookings WITH prior cancellations, by half-year:
2015 H2   99.0%
2016 H1   89.2%



Train rows with prior cancellations: 5,809
  in an identical-row group: 86%
  marked 'not a repeated guest': 91%
  Non Refund deposit: 59% (all train bookings: 15%)


**Options considered**

| Option | Verdict |
|---|---|
| A. Use it unchanged, no test | **Rejected.** The pattern weakens inside train and almost disappears later; a model could rely on it and lose accuracy exactly where it is scored. |
| B. Drop it now | **Rejected as premature.** It still separates outcomes strongly in train's later half. Dropping it before testing would be a guess; the choose period exists to test exactly this. |
| C. Recompute, cap or binarise the counts | **Rejected.** No customer ID to recompute from, and capping or binarising does not fix drift. |
| D. Change values that look like group counting | **Rejected.** The values may be accurate records; no source says otherwise (P2). |
| E. **No value change; in Step 5 compare every candidate with and without it on choose** | **Recommended.** Notebook 03 **C6**; notebook 02 gate **D4**. |

**Reasoning chain**

1. Share of bookings with prior cancellations falls from 23.8% (2015 Q3) to under 1% in 2017. Train has 11.8%; choose has 1.3%.
2. Inside train, bookings with prior cancellations cancelled 99.0% of the time in 2015 H2 and 89.2% in 2016 H1: still predictive, but weakening.
3. Most of these rows sit in identical-row groups, are marked "not a repeated guest", and are far more often Non Refund than train overall. One possible reading is that the field sometimes counted cancelled entries from the same group block. That is a **hypothesis**; the source does not confirm it.
4. We can't prove the values are wrong, so we don't change them (P2).
5. Whether the model should use the field is a method question, and rule 2 says such choices are made on choose and frozen before proof. So: test it on choose.

**Challenges and answers**

- *"Isn't showing the 2017 share looking at proof?"* Rule 2 forbids using proof **outcomes**. The quarterly share is a feature distribution, read without any outcome. The decision still does not rest on it: train and choose alone (11.8% vs 1.3%) lead to the same test.
- *"If choose picks 'without', haven't you selected a feature on choose?"* Yes, and that is what choose is for. The choice is frozen before the single proof run.
- *"Why not fix the group counting?"* We can't tell which counts are inflated without a customer or booking ID. Changing values on a hypothesis would be inventing data.
- *"The flag `flag_repeat_status_conflict` marks 5,571 train rows. Is `is_repeated_guest` broken too?"* The source defines it differently: it needs a customer profile created before the booking, while the counts come from the booking records. So they can disagree honestly. It is flagged as a known inconsistency, not corrected.

**What would change this decision:** a source explanation of how the counts were built (then a recompute might be justified), or choose-period results showing the field doesn't matter either way.

**Say it in one sentence:** "previous_cancellations still predicts in train but almost vanishes by the choose period, and I can't prove the values are wrong, so I don't change them; I test every candidate with and without the field on choose and freeze that choice before proof."

## Step 12. How the decisions fit together

The decisions are consistent because they all follow the same principles:

| Step | Recommendation | Principle it rests on | Notebook 03 | Key number | If you disapprove |
|---|---|---|---|---|---|
| 1 | Keep repeated rows; flag | P2, P3, P4 | C5 | Train rate 35.8% → 22.2% if dropped | Pause until a booking key or dedup rule is agreed |
| 2 | Keep zero-night bookings; flag | P3, P4 | C5 | 354 of 371 train ones checked out | Needs a client change to the book definition |
| 3 | Keep zero-guest / `P` / no-adult rows; flag | P2, P3 | C5 | 45 of 49 train zero-guest rows checked out | State the rule that proves them invalid |
| 4 | Keep 10+ adult rows; flag | P5 | C5 | Rate moves by about 0.02 points | State a non-outcome rule for removal |
| 5 | Keep ADR outliers; flag | P2, P3 | C5 | V moves EUR 28 (train) and under EUR 1 (proof) | Need a source value or a train-fitted rule |
| 6 | `Undefined` → `SC` | P2 (source definition) | C1 | 1,169 rows | Keep both codes; note they mean the same |
| 7 | IDs as text; NULL → `none` | P2 (source definition) | C2 | agent 16,340 / company 112,593 NULL | Keep decimals; avoid using IDs numerically |
| 8 | Missing country → `unknown`; keep `CN` | P2 | C3 | 488 missing; 1,279 `CN` | Leave `NaN`; no effect on the MVP |
| 9 | 4 missing children → 0; flag | P2, traceability | C4 | 95.1% of train have 0 | Leave `NaN` |
| 10 | No action | — | — | — | — |
| 11 | No value change; test on choose | P2, rule 2 | C6 | 11.8% → 1.3% train to choose | Pause the feature decision |

**What all twelve add up to:** no row is removed, the contract's scoring inputs (`hotel`, `arrival_date`, `adr`, `nights`) are unchanged, and every odd row is flagged. Notebook 03, section 11, checks this with assertions when the cleaned table is built.

### The three sentences that summarise all of it

1. "I changed a value only where the dataset article defines what it should be, and I removed no rows, because without a booking ID nothing proves a row is an error."
2. "The contract scores every booking due to arrive, so removing rows would change the client's scoring and train the model on a different population from the one it is judged on."
3. "The real risk was a feature, not dirty data: `previous_cancellations` fades out after 2015, so I test it with and without on choose instead of trusting it or dropping it blindly."

## What to record after you decide

`src/data.py` asks that every cleaning choice be recorded in `docs/DECISIONS.md` with its reason and row count. After you set your choices in notebook 03, section 10, these draft rows can be adapted. **Change the status to what you actually decided**; they are not filled in here because your decisions are still pending.

| Date | Choice | Evidence / reason | Rows touched | Status |
|---|---|---|---:|---|
| (date) | C5: keep all exact repeats, zero-night, zero-guest, `P`, 10+ adult and ADR-outlier rows; add `flag_` columns | No booking ID; look-alike groups with different outcomes; removal changes the train rate (35.8% → 22.2%) and the contract's book | 0 removed; flags only | (your decision) |
| (date) | C1: `meal` `Undefined` → `SC` | Article: "Undefined/SC – no meal package" | 1,169 recoded | (your decision) |
| (date) | C2: `agent` / `company` as text; NULL → `none` | Article: NULL means not applicable; IDs are labels | agent 16,340, company 112,593 labelled `none`; all IDs retyped | (your decision) |
| (date) | C3: missing `country` → `unknown`; `CN` kept separate | Bin 3; `CN` meaning unconfirmed | 488 labelled | (your decision) |
| (date) | C4: missing `children` → 0 with flag | 95.1% of train have 0; 4 rows in one incomplete cluster | 4 filled | (your decision) |
| (date) | C6: no change to `previous_cancellations`; test with/without on choose | Share falls 11.8% (train) → 1.3% (choose); weakening inside train | 0 changed | (your decision) |

## Self-test: answer aloud before opening

<details><summary>1. Why didn't you remove the duplicate rows?</summary>

No booking ID proves they are copies; some look-alike groups had different outcomes; they look like multi-room group bookings; removing them drops the train rate from 35.8% to 22.2% while they are still about 23% of the scored book.
</details>

<details><summary>2. Isn't keeping zero-night bookings wrong, since they don't stay overnight?</summary>

Most checked out, so they are real. The contract counts every booking due to arrive; removing them would change V and the book. Whether they should count is a client question.
</details>

<details><summary>3. What is the difference between flagging and cleaning?</summary>

A flag records an observation and changes nothing. Cleaning changes a value or removes a row and needs a source rule and a log entry.
</details>

<details><summary>4. Why not remove the 13 bookings with 10+ adults?</summary>

They are train-only, move the rate by about 0.02 points, and what singles them out is mainly that they all cancelled. Removing rows because of their outcome biases the target.
</details>

<details><summary>5. What justifies recoding Undefined to SC?</summary>

The dataset article defines them as the same "no meal package" category. That source definition meets the bar for changing a value.
</details>

<details><summary>6. Why not merge CN into CHN?</summary>

Probably the same country, but no source confirms it, both codes appear side by side throughout, and country is never a predictor.
</details>

<details><summary>7. Did any cleaning decision use proof outcomes?</summary>

No. Proof was used only for structural counts and two non-outcome detail tables. Outcome-based checks ran on train only.
</details>

<details><summary>8. What is the biggest risk you found, and what did you do?</summary>

`previous_cancellations` drifts from 11.8% of train bookings to 1.3% of choose. No values changed; every candidate is compared with and without it on choose, and the choice is frozen before proof.
</details>

<details><summary>9. What is still not right after cleaning?</summary>

Group bookings cancel together, so the contract's `spread` (which assumes independence) understates uncertainty. Address it when choosing the buffer on choose. Same-day stays in the book are a client question.
</details>

<details><summary>10. What evidence would make you remove rows after all?</summary>

A booking ID or a client or system rule showing specific rows are copies or placeholders. Then remove them in both training and the scored book, by a stated rule, and log the count.
</details>

## What this notebook does not do

- It does not change data, build a cleaned table, or record your decisions. Decisions go in notebook 03, section 10.
- It does not read any choose or proof outcome, train a model, or claim any result.
- The hypotheses it names (multi-room group exports, `P` as a placeholder, group-counted prior cancellations) are labelled as hypotheses. The source does not confirm them.